# TP - Regularización en la clasificación de 20 Newsgroups (TNG)
### Redes Neuronales / Deep Learning

Este notebook continúa `TNG+MLP.ipynb`. Allí vimos que la red con capas ocultas **sobreajusta** (train ≈ 0.96 vs. val ≈ 0.60). Acá incorporamos como hiperparámetros las técnicas de regularización vistas en clase:

| Técnica | Hiperparámetro(s) |
|---|---|
| Regularización **L1**, **L2** y **L1+L2** | `l1`, `l2` (coeficientes λ de la penalidad sobre los pesos) |
| **Dropout** | `dropout` (probabilidad `p` de apagar cada neurona) |
| **Batch Normalization** | `batchnorm` (sí/no), y su interacción con el `lr` |

## Qué medimos en cada experimento
Siguiendo las recomendaciones de clase, para cada corrida no miramos solo la accuracy final:

1. **Valores absolutos de los pesos**: `mean|w|`, `max|w|` y % de pesos casi nulos (`|w| < 1e-3`), más el histograma de `|w|`.
2. **Tiempos de entrenamiento**: segundos por época (solo la pasada de entrenamiento) y épocas necesarias para llegar a una accuracy objetivo.
3. **Learning rate**: barrido de `lr` con y sin BatchNorm.
4. **Ruido en las curvas de aprendizaje**: desvío de la accuracy de validación respecto de su media móvil.
5. **Brecha de sobreajuste**: `train_acc - val_acc`, con el train evaluado en modo `eval()` (así Dropout y BN no distorsionan la comparación).

## Protocolo
- Preprocesamiento y vectorización **fijos** (los mejores de la búsqueda anterior), para que lo único que cambie sea la regularización.
- Arquitectura **fija** `[32, 128, 128, 64]` con ReLU: la que encontramos que más sobreajusta. Es el escenario donde regularizar tiene más para aportar.
- Se varía **un factor por vez**, con la misma semilla y el mismo split train/validación.
- El **test set se usa una sola vez**, al final, para comparar el modelo base contra el mejor configurado.
- Al final, una búsqueda con **Optuna** que combina todos los hiperparámetros.

## 0. Setup

In [ ]:
!pip install -q mlflow optuna

In [ ]:
import re, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import mlflow
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

## 1. Datos y features (fijos)

Usamos los mejores hiperparámetros de la búsqueda de `TNG+MLP.ipynb`: sin quitar stopwords, stemming, unigramas, `max_features=10000`, `min_df=1`.

Nota: `TfidfVectorizer` ya devuelve cada documento normalizado en L2 (`norm="l2"` es su valor por defecto), así que el `Normalizer` extra que usábamos antes era redundante y acá lo omitimos.

In [ ]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

newsgroups_train = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
newsgroups_test  = fetch_20newsgroups(subset="test",  remove=("headers", "footers", "quotes"))
class_names = newsgroups_train.target_names
NUM_CLASSES = len(class_names)
print(f"Train: {len(newsgroups_train.data)} docs | Test: {len(newsgroups_test.data)} docs | Clases: {NUM_CLASSES}")

STEMMER = PorterStemmer()

def preprocess_text(text):
    tokens = [t for t in word_tokenize(text.lower()) if t.isalpha()]
    return " ".join(STEMMER.stem(t) for t in tokens)

print("Preprocesando (puede tardar un par de minutos)...")
train_proc = [preprocess_text(t) for t in newsgroups_train.data]
test_proc  = [preprocess_text(t) for t in newsgroups_test.data]

In [ ]:
def build_features(fit_texts, other_texts):
    vec = TfidfVectorizer(ngram_range=(1, 1), max_features=10000, min_df=1)
    X_fit = vec.fit_transform(fit_texts)
    X_other = vec.transform(other_texts)
    return X_fit, X_other

def to_tensors(X, y):
    return (torch.tensor(X.toarray(), dtype=torch.float32),
            torch.tensor(np.asarray(y), dtype=torch.long))

# --- Datos para los experimentos: train (80%) / validación (20%) sobre el train original
tr_texts, val_texts, y_tr, y_val = train_test_split(
    train_proc, newsgroups_train.target, test_size=0.2, random_state=SEED,
    stratify=newsgroups_train.target)
X_tr, X_val = build_features(tr_texts, val_texts)
DATA_EXP = {}
DATA_EXP["X_tr"], DATA_EXP["y_tr"] = to_tensors(X_tr, y_tr)
DATA_EXP["X_ev"], DATA_EXP["y_ev"] = to_tensors(X_val, y_val)
INPUT_DIM = DATA_EXP["X_tr"].shape[1]
print("Train:", tuple(DATA_EXP["X_tr"].shape), "| Validación:", tuple(DATA_EXP["X_ev"].shape))

## 2. Modelo configurable

Un MLP donde cada bloque oculto es `Linear -> [BatchNorm1d] -> ReLU -> [Dropout]`:

- **BatchNorm antes de la activación**, que es la ubicación del paper original. Cuando hay BN, el `bias` de la capa lineal es redundante (BN ya resta la media y agrega su propio `β`), así que se desactiva.
- **Dropout después de la activación**. Se usa *inverted dropout*: en entrenamiento se escala por `1/(1-p)`, de modo que en `eval()` no hay que reescalar nada.
- **L1/L2 se agregan a la loss** como penalidad explícita sobre los **pesos de las capas lineales**, sin incluir biases ni parámetros de BN (`γ`, `β`), como se recomienda en clase. Convención: `L2 = λ₂·Σw²` y `L1 = λ₁·Σ|w|`.

Se implementan como penalidad explícita en la loss y no con `weight_decay` del optimizador por dos motivos: permite combinar L1+L2 con la misma maquinaria, y con Adam el `weight_decay` equivale a L2 *acoplado* al escalado adaptativo del gradiente, lo que complica comparar valores de λ entre optimizadores.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_units, dropout=0.0, batchnorm=False):
        super().__init__()
        layers, d = [], input_dim
        for h in hidden_units:
            layers.append(nn.Linear(d, h, bias=not batchnorm))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h
        layers.append(nn.Linear(d, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def reg_penalty(model, l1, l2):
    # Penalidad sólo sobre los pesos de las capas Linear (no biases, no BN).
    pen = 0.0
    if l1 == 0 and l2 == 0:
        return pen
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if l1 > 0:
                pen = pen + l1 * m.weight.abs().sum()
            if l2 > 0:
                pen = pen + l2 * (m.weight ** 2).sum()
    return pen


def linear_weights_abs(model):
    return torch.cat([m.weight.detach().abs().flatten() for m in model.modules() if isinstance(m, nn.Linear)])


def weight_stats(model, thr=1e-3):
    w = linear_weights_abs(model)
    return w.mean().item(), w.max().item(), (w < thr).float().mean().item()

## 3. Entrenamiento y métricas

`run_experiment` entrena un modelo y guarda, **por época**: loss/accuracy de train (en modo `eval`) y de validación, `mean|w|`, `max|w|`, % de pesos casi nulos y tiempo de la pasada de entrenamiento. Loguea en TensorBoard y MLflow.

Métricas resumen:
- **Ruido** de la curva: desvío estándar de `val_acc - media_móvil_3(val_acc)`. Mide cuánto zigzaguea la curva alrededor de su tendencia (a partir de la época 2).
- **Épocas a objetivo**: primera época en que `val_acc ≥ TARGET_ACC` (indicador de velocidad de convergencia).
- **Brecha**: `train_acc_eval - val_acc` en la última época.

In [ ]:
BASE = dict(hidden=[32, 128, 128, 64], optimizer="Adam", lr=1e-3, batch_size=32, epochs=20,
            l1=0.0, l2=0.0, dropout=0.0, batchnorm=False)
TARGET_ACC = 0.60
RESULTS = {}

mlflow.set_experiment("regularizacion_20newsgroups")


def make_loader(X, y, batch_size, shuffle, seed=SEED):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle,
                      generator=g if shuffle else None)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    crit = nn.CrossEntropyLoss(reduction="sum")
    loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss += crit(out, yb).item()
        correct += (out.argmax(1) == yb).sum().item()
        n += xb.size(0)
    return loss / n, correct / n


def curve_noise(v):
    v = np.asarray(v, dtype=float)
    if len(v) < 4:
        return float("nan")
    ma = (v[:-2] + v[1:-1] + v[2:]) / 3.0
    return float(np.std(v[1:-1] - ma))


def make_optimizer(name, params, lr):
    if name == "Adam":
        return optim.Adam(params, lr=lr)
    if name == "SGD":
        return optim.SGD(params, lr=lr, momentum=0.9)
    if name == "RMSprop":
        return optim.RMSprop(params, lr=lr)
    raise ValueError(name)


def run_experiment(name, group, data=None, trial=None, keep_weights=True, keep_model=False,
                   verbose=True, tb=True, **overrides):
    cfg = {**BASE, **overrides}
    data = data or DATA_EXP
    set_seed(SEED)

    model = MLP(INPUT_DIM, NUM_CLASSES, cfg["hidden"], cfg["dropout"], cfg["batchnorm"]).to(device)
    opt = make_optimizer(cfg["optimizer"], model.parameters(), cfg["lr"])
    crit = nn.CrossEntropyLoss()
    train_loader = make_loader(data["X_tr"], data["y_tr"], cfg["batch_size"], True)
    train_eval_loader = make_loader(data["X_tr"], data["y_tr"], 1024, False)
    ev_loader = make_loader(data["X_ev"], data["y_ev"], 1024, False)

    writer = SummaryWriter(log_dir=f"runs/reg/{group}/{name}") if tb else None
    hist = {k: [] for k in ["train_loss_run", "train_loss", "train_acc", "val_loss", "val_acc",
                            "epoch_time", "mean_abs_w", "max_abs_w", "frac_small_w"]}

    with mlflow.start_run(run_name=name, nested=True):
        mlflow.log_params({k: str(v) for k, v in cfg.items()})
        mlflow.log_param("group", group)

        for epoch in range(cfg["epochs"]):
            model.train()
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            run_loss, n = 0.0, 0
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                ce = crit(model(xb), yb)
                (ce + reg_penalty(model, cfg["l1"], cfg["l2"])).backward()
                opt.step()
                run_loss += ce.item() * xb.size(0)
                n += xb.size(0)
            if device.type == "cuda":
                torch.cuda.synchronize()
            epoch_time = time.time() - t0

            tr_loss, tr_acc = evaluate(model, train_eval_loader)
            va_loss, va_acc = evaluate(model, ev_loader)
            mw, xw, fs = weight_stats(model)

            for k, v in zip(hist.keys(), [run_loss / n, tr_loss, tr_acc, va_loss, va_acc, epoch_time, mw, xw, fs]):
                hist[k].append(v)

            if writer:
                writer.add_scalars("loss", {"train": tr_loss, "val": va_loss}, epoch)
                writer.add_scalars("accuracy", {"train": tr_acc, "val": va_acc}, epoch)
                writer.add_scalar("weights/mean_abs", mw, epoch)
                writer.add_scalar("time/epoch_s", epoch_time, epoch)
            mlflow.log_metrics({"train_acc": tr_acc, "val_acc": va_acc, "train_loss": tr_loss, "val_loss": va_loss,
                                "mean_abs_w": mw, "epoch_time": epoch_time}, step=epoch)

            if verbose and ((epoch + 1) % 5 == 0 or epoch == 0 or epoch + 1 == cfg["epochs"]):
                print(f"[{name}] ep {epoch+1:>2}/{cfg['epochs']} train_acc {tr_acc:.3f} val_acc {va_acc:.3f} "
                      f"val_loss {va_loss:.3f} mean|w| {mw:.5f} t/ep {epoch_time:.1f}s")

            if trial is not None:
                trial.report(va_acc, epoch)
                if trial.should_prune():
                    if writer: writer.close()
                    raise optuna.TrialPruned()

        w_abs = linear_weights_abs(model).cpu().numpy() if keep_weights else None
        if writer:
            writer.add_histogram("weights/abs", w_abs if w_abs is not None else linear_weights_abs(model).cpu().numpy(),
                                 cfg["epochs"])
            writer.close()

    res = dict(name=name, group=group, cfg=cfg, hist=hist, w_abs=w_abs,
               model=model if keep_model else None,
               total_time=float(np.sum(hist["epoch_time"])))
    if trial is None:
        RESULTS[name] = res
    return res


def summarize(names):
    rows = []
    for n in names:
        r = RESULTS[n]; h = r["hist"]
        va = np.array(h["val_acc"])
        reached = np.where(va >= TARGET_ACC)[0]
        rows.append({
            "run": n,
            "best_val_acc": va.max(),
            "best_epoch": int(va.argmax()) + 1,
            "final_val_acc": va[-1],
            "final_val_loss": h["val_loss"][-1],
            "final_train_acc": h["train_acc"][-1],
            "brecha (train-val)": h["train_acc"][-1] - va[-1],
            "mean|w|": h["mean_abs_w"][-1],
            "max|w|": h["max_abs_w"][-1],
            "% |w|<1e-3": 100 * h["frac_small_w"][-1],
            "s/epoca": float(np.mean(h["epoch_time"])),
            f"epocas a {TARGET_ACC:.0%}": int(reached[0]) + 1 if len(reached) else np.nan,
            "ruido val_acc": curve_noise(va),
        })
    return pd.DataFrame(rows).set_index("run").round(4)


def plot_group(names, title):
    fig, axs = plt.subplots(2, 3, figsize=(17, 8))
    panels = [("val_acc", "Accuracy de validación"), ("val_loss", "Loss de validación"),
              ("train_acc", "Accuracy de train (modo eval)"), ("mean_abs_w", "mean |w| (capas lineales)"),
              ("frac_small_w", "Fracción de pesos con |w| < 1e-3")]
    for ax, (key, ttl) in zip(axs.flat, panels):
        for n in names:
            ax.plot(range(1, len(RESULTS[n]["hist"][key]) + 1), RESULTS[n]["hist"][key], label=n,
                    lw=2.2 if n == "baseline" else 1.4, color="black" if n == "baseline" else None)
        ax.set_title(ttl); ax.set_xlabel("época"); ax.grid(alpha=.3)
    # sexto panel: brecha
    ax = axs.flat[5]
    for n in names:
        h = RESULTS[n]["hist"]
        ax.plot(range(1, len(h["val_acc"]) + 1), np.array(h["train_acc"]) - np.array(h["val_acc"]), label=n,
                lw=2.2 if n == "baseline" else 1.4, color="black" if n == "baseline" else None)
    ax.set_title("Brecha train - val (accuracy)"); ax.set_xlabel("época"); ax.grid(alpha=.3)
    axs.flat[0].legend(fontsize=8)
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()


def plot_weight_hist(names, title):
    fig, ax = plt.subplots(figsize=(9, 4))
    for n in names:
        w = RESULTS[n]["w_abs"]
        ax.hist(w, bins=np.logspace(-8, 0, 80), histtype="step", lw=1.6, label=n)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("|w| (escala log)"); ax.set_ylabel("cantidad de pesos")
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.show()


def best_of(group):
    cands = [n for n, r in RESULTS.items() if r["group"] == group]
    return max(cands, key=lambda n: RESULTS[n]["hist"]["val_acc"][-1])

## 4. Línea base (sin regularización)

Todas las comparaciones se hacen contra esta corrida (línea negra en los gráficos). Es la arquitectura que sobreajusta, con `Adam` y `lr=1e-3`.

In [ ]:
run_experiment("baseline", "baseline")
summarize(["baseline"])

## 5. Regularización L2

**Qué esperamos (teoría):** la penalidad `λ·Σw²` agrega `2λw` al gradiente: cada paso encoge los pesos en proporción a su tamaño (*weight decay*). Equivale a una *prior* gaussiana sobre los pesos (MAP). Los pesos se achican pero **no se anulan**: la distribución de `|w|` se corre hacia valores chicos sin acumular masa en cero. Un `λ` muy chico no cambia nada y uno muy grande produce *underfitting* (cae también la accuracy de train).

**Recomendaciones aplicadas:** barrer `λ` en **escala logarítmica**, penalizar solo pesos (no biases/BN) y mirar el efecto sobre `mean|w|` y la brecha train-val.

In [ ]:
l2_names = []
for l2 in [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]:
    n = f"L2={l2:g}"
    run_experiment(n, "L2", l2=l2, verbose=False)
    l2_names.append(n)
plot_group(["baseline"] + l2_names, "Efecto de la regularización L2")
plot_weight_hist(["baseline"] + l2_names, "Distribución de |w| - L2")
summarize(["baseline"] + l2_names)

**Observaciones L2** *(completar con lo observado en las tablas y gráficos de arriba)*:
- Valor de λ que mejor equilibra val_acc y brecha: `<completar>`
- Cambio en `mean|w|` y `max|w|` respecto del baseline: `<completar>`
- ¿Aparecen pesos exactamente nulos? `<completar>` (esperado: no, solo más chicos)
- Con λ demasiado alto: `<completar>` (esperado: underfitting, cae también train_acc)
- Tiempo por época: `<completar>` (esperado: casi igual, la penalidad es barata)

## 6. Regularización L1

**Qué esperamos (teoría):** la penalidad `λ·Σ|w|` aporta `λ·sign(w)` al gradiente, una fuerza **de magnitud constante** hacia cero sin importar cuán chico sea el peso. Equivale a una *prior* de Laplace y favorece soluciones **ralas** (*sparse*): muchos pesos en (casi) cero. En la primera capa, cada peso conecta una palabra del vocabulario con una neurona, así que L1 actúa como **selección de palabras**.

Con SGD/Adam los pesos no quedan *exactamente* en 0 (oscilan alrededor de cero porque el subgradiente cambia de signo), por eso medimos la fracción con `|w| < 1e-3`. Esa oscilación también puede agregar **ruido** a las curvas.

Como la suma `Σ|w|` crece con la cantidad de pesos (la primera capa tiene 320.000), el rango útil de λ es más bajo que en L2.

In [ ]:
l1_names = []
for l1 in [1e-6, 1e-5, 1e-4, 1e-3]:
    n = f"L1={l1:g}"
    run_experiment(n, "L1", l1=l1, verbose=False)
    l1_names.append(n)
plot_group(["baseline"] + l1_names, "Efecto de la regularización L1")
plot_weight_hist(["baseline"] + l1_names, "Distribución de |w| - L1")
summarize(["baseline"] + l1_names)

**Observaciones L1** *(completar)*:
- λ que mejor equilibra val_acc y brecha: `<completar>`
- % de pesos con |w| < 1e-3 vs. L2 y vs. baseline: `<completar>` (esperado: mucho mayor, histograma con masa cerca de cero)
- Ruido de la curva vs. baseline: `<completar>`
- Con λ alto: `<completar>`

## 7. L1 + L2 (Elastic Net)

**Qué esperamos (teoría):** combina la **esparsidad** de L1 con la **estabilidad** de L2 (que reparte el peso entre variables correlacionadas en lugar de elegir una al azar, algo habitual entre palabras muy correlacionadas). Ahora hay dos hiperparámetros, y el espacio a explorar crece: por eso conviene una grilla chica o una búsqueda automática (sección 11).

In [ ]:
l12_names = []
for l1 in [1e-5, 1e-4]:
    for l2 in [1e-4, 1e-3]:
        n = f"L1={l1:g}+L2={l2:g}"
        run_experiment(n, "L1+L2", l1=l1, l2=l2, verbose=False)
        l12_names.append(n)

ref = ["baseline", best_of("L1"), best_of("L2")]
plot_group(ref + l12_names, "L1+L2 vs. mejor L1, mejor L2 y baseline")
plot_weight_hist(ref + l12_names, "Distribución de |w| - L1+L2")
summarize(ref + l12_names)

**Observaciones L1+L2** *(completar)*:
- ¿Mejora a la mejor L1 o la mejor L2 por separado? `<completar>`
- Combinación de esparsidad y magnitud de pesos: `<completar>`

## 8. Dropout

**Qué esperamos (teoría):** en cada paso se apaga cada neurona con probabilidad `p`, lo que impide la **co-adaptación** entre neuronas y entrena implícitamente un *ensemble* de subredes que se promedia en inferencia (`eval()` usa la red completa). Consecuencias esperadas:
- La loss de **entrenamiento medida en modo train** sube y es más ruidosa (cada paso usa otra subred); por eso comparamos con el train evaluado en `eval()`.
- **Converge más lento**: hace falta más épocas para llegar a la misma accuracy.
- Valores típicos `p ∈ [0.2, 0.5]` en capas ocultas. Acá la primera capa oculta tiene solo **32 neuronas**, así que un `p` alto (0.5-0.7) puede dejar un cuello de botella y causar *underfitting*.
- No necesariamente achica `|w|`: regulariza a través del ruido, no de la norma de los pesos.

In [ ]:
do_names = []
for p in [0.1, 0.2, 0.3, 0.5, 0.7]:
    n = f"dropout={p:g}"
    run_experiment(n, "Dropout", dropout=p, verbose=False)
    do_names.append(n)
plot_group(["baseline"] + do_names, "Efecto de Dropout")
plot_weight_hist(["baseline"] + do_names, "Distribución de |w| - Dropout")
summarize(["baseline"] + do_names)

In [ ]:
# Dropout: loss de train medida durante el entrenamiento (con máscara activa) vs. en modo eval
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for n in ["baseline", best_of("Dropout"), "dropout=0.5"]:
    h = RESULTS[n]["hist"]
    axs[0].plot(h["train_loss_run"], label=f"{n} (train mode)")
    axs[0].plot(h["train_loss"], "--", label=f"{n} (eval mode)")
axs[0].set_title("Loss de train: durante el entrenamiento vs. en eval()"); axs[0].set_xlabel("época")
axs[0].set_yscale("log"); axs[0].legend(fontsize=7); axs[0].grid(alpha=.3)
for n in ["baseline"] + do_names:
    axs[1].bar(n, RESULTS[n]["hist"]["val_acc"][-1])
axs[1].set_title("Accuracy de validación final"); axs[1].tick_params(axis="x", rotation=45)
fig.tight_layout(); plt.show()

**Observaciones Dropout** *(completar)*:
- `p` óptimo: `<completar>`; con `p=0.7`: `<completar>` (esperado: underfitting por la capa de 32 neuronas)
- Diferencia entre loss de train en modo train y en eval: `<completar>`
- Épocas a la accuracy objetivo vs. baseline: `<completar>`
- Ruido de la curva de validación: `<completar>`
- `mean|w|` vs. baseline: `<completar>`

## 9. Batch Normalization y learning rate

**Qué esperamos (teoría):** BN normaliza las pre-activaciones de cada capa con la media y varianza del mini-batch, y luego las reescala con parámetros aprendibles `γ` y `β`. Efectos esperados:
- **Permite learning rates más altos** y hace al entrenamiento menos sensible a la inicialización: mejora el condicionamiento y suaviza la superficie de la loss.
- **Converge en menos épocas**, pero **cada época cuesta más** (operaciones y parámetros extra).
- Agrega **ruido** por las estadísticas del mini-batch, y con `batch_size=32` ese ruido es apreciable. Funciona como regularizador leve, y a la vez las curvas pueden ser más irregulares.
- Hay diferencia entre `train()` (estadísticas del batch) y `eval()` (promedios móviles).
- Con BN, la escala de los pesos antes de la normalización no altera la salida (invarianza de escala): por eso `|w|` pierde interpretación como medida de capacidad, y L2 pasa a controlar sobre todo el *learning rate efectivo*.

**Diseño:** barrido de `lr` ∈ {1e-4, 1e-3, 1e-2, 3e-2} con y sin BN, todo lo demás igual.

In [ ]:
bn_names = []
for lr in [1e-4, 1e-3, 1e-2, 3e-2]:
    for bn in [False, True]:
        n = f"lr={lr:g}|BN={'si' if bn else 'no'}"
        run_experiment(n, "BN_LR", lr=lr, batchnorm=bn, verbose=False)
        bn_names.append(n)

no_bn = [n for n in bn_names if "BN=no" in n]
si_bn = [n for n in bn_names if "BN=si" in n]
plot_group(no_bn, "Sin BatchNorm: barrido de learning rate")
plot_group(si_bn, "Con BatchNorm: barrido de learning rate")
summarize(bn_names)

In [ ]:
# Resumen lr x BN
tab = summarize(bn_names)[["final_val_acc", "brecha (train-val)", "ruido val_acc", "s/epoca", "mean|w|"]].reset_index()
tab["lr"] = tab["run"].str.extract(r"lr=([^|]+)")[0].astype(float)
tab["BN"] = tab["run"].str.extract(r"BN=(\w+)")[0]

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for bn, g in tab.groupby("BN"):
    g = g.sort_values("lr")
    axs[0].plot(g["lr"], g["final_val_acc"], "o-", label=f"BN={bn}")
    axs[1].plot(g["lr"], g["ruido val_acc"], "o-", label=f"BN={bn}")
    axs[2].plot(g["lr"], g["s/epoca"], "o-", label=f"BN={bn}")
for ax, t in zip(axs, ["Accuracy de validación final", "Ruido de la curva de validación", "Segundos por época"]):
    ax.set_xscale("log"); ax.set_xlabel("learning rate"); ax.set_title(t); ax.grid(alpha=.3); ax.legend()
fig.tight_layout(); plt.show()

**Observaciones BN / LR** *(completar)*:
- Rango de `lr` que funciona sin BN vs. con BN: `<completar>`
- ¿BN con `lr` alto converge en menos épocas? `<completar>`
- Costo en segundos por época: `<completar>`
- Ruido de las curvas con BN (batch de 32): `<completar>`
- Brecha train-val con BN: `<completar>` (¿regulariza por sí solo?)

## 10. Búsqueda conjunta con Optuna

Ahora la regularización pasa a ser parte del espacio de búsqueda. Optuna decide el tipo de penalidad (`none`, `l1`, `l2`, `l1_l2`), sus coeficientes, `dropout`, `batchnorm` y `lr`. Mantenemos fijos el preprocesamiento y la arquitectura. Se optimiza la accuracy de validación en la última época.

Rangos (escala logarítmica para λ y `lr`, como se recomienda): `l1 ∈ [1e-7, 1e-3]`, `l2 ∈ [1e-6, 1e-1]`, `dropout ∈ [0, 0.6]`, `lr ∈ [1e-4, 3e-2]`.

In [ ]:
def objective(trial):
    reg = trial.suggest_categorical("reg", ["none", "l1", "l2", "l1_l2"])
    l1 = trial.suggest_float("l1", 1e-7, 1e-3, log=True) if reg in ("l1", "l1_l2") else 0.0
    l2 = trial.suggest_float("l2", 1e-6, 1e-1, log=True) if reg in ("l2", "l1_l2") else 0.0
    dropout = trial.suggest_float("dropout", 0.0, 0.6, step=0.1)
    batchnorm = trial.suggest_categorical("batchnorm", [False, True])
    lr = trial.suggest_float("lr", 1e-4, 3e-2, log=True)

    res = run_experiment(f"optuna_{trial.number}", "optuna", trial=trial, epochs=12,
                         keep_weights=False, verbose=False, tb=False,
                         l1=l1, l2=l2, dropout=dropout, batchnorm=batchnorm, lr=lr)
    trial.set_user_attr("mean_abs_w", res["hist"]["mean_abs_w"][-1])
    trial.set_user_attr("s_por_epoca", float(np.mean(res["hist"]["epoch_time"])))
    trial.set_user_attr("brecha", res["hist"]["train_acc"][-1] - res["hist"]["val_acc"][-1])
    return res["hist"]["val_acc"][-1]

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3))
study.optimize(objective, n_trials=25)

print("Mejores hiperparámetros:", study.best_params)
print("Mejor accuracy de validación:", round(study.best_value, 4))

In [ ]:
df_trials = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
df_trials = df_trials.sort_values("value", ascending=False)
df_trials.head(10)

In [ ]:
try:
    imp = optuna.importance.get_param_importances(study)
    print("Importancia de cada hiperparámetro:")
    for k, v in imp.items():
        print(f"  {k:10s} {v:.3f}")
except Exception as e:
    print("No se pudo calcular la importancia:", e)

done = df_trials[df_trials["state"] == "COMPLETE"]
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for bn, g in done.groupby("params_batchnorm"):
    axs[0].scatter(g["params_lr"], g["value"], label=f"BN={bn}", s=50)
axs[0].set_xscale("log"); axs[0].set_xlabel("learning rate"); axs[0].set_ylabel("val_acc final")
axs[0].set_title("Trials: lr vs. accuracy (según BN)"); axs[0].legend(); axs[0].grid(alpha=.3)
for reg, g in done.groupby("params_reg"):
    axs[1].scatter(g["params_dropout"], g["value"], label=reg, s=50)
axs[1].set_xlabel("dropout"); axs[1].set_ylabel("val_acc final")
axs[1].set_title("Trials: dropout vs. accuracy (según tipo de L1/L2)"); axs[1].legend(); axs[1].grid(alpha=.3)
fig.tight_layout(); plt.show()

**Observaciones Optuna** *(completar)*:
- Combinación ganadora y qué técnicas incluye: `<completar>`
- Hiperparámetro más importante: `<completar>`
- ¿Coincide con lo visto en los barridos de a un factor? `<completar>`
- Cuántos trials fueron podados y en qué tipo de configuraciones: `<completar>`

## 11. Entrenamiento final y evaluación en test

Reentrenamos con **todo** el train original, tanto el modelo base como el mejor configurado por Optuna, y evaluamos contra el **test set**, que no se usó hasta ahora. Se reporta la **última época** (no se elige la época mirando el test).

In [ ]:
EPOCHS_FINAL = 20

X_full, X_test = build_features(train_proc, test_proc)
DATA_FULL = {}
DATA_FULL["X_tr"], DATA_FULL["y_tr"] = to_tensors(X_full, newsgroups_train.target)
DATA_FULL["X_ev"], DATA_FULL["y_ev"] = to_tensors(X_test, newsgroups_test.target)

bp = study.best_params
BEST_CFG = dict(
    l1=bp.get("l1", 0.0), l2=bp.get("l2", 0.0), dropout=bp["dropout"],
    batchnorm=bp["batchnorm"], lr=bp["lr"])
print("Config final:", BEST_CFG)

res_base = run_experiment("final_baseline", "final", data=DATA_FULL, epochs=EPOCHS_FINAL, keep_model=True)
res_best = run_experiment("final_best", "final", data=DATA_FULL, epochs=EPOCHS_FINAL, keep_model=True, **BEST_CFG)

In [ ]:
@torch.no_grad()
def predict(model, X):
    model.eval()
    preds = []
    for i in range(0, len(X), 1024):
        preds.append(model(X[i:i + 1024].to(device)).argmax(1).cpu())
    return torch.cat(preds).numpy()

y_test = DATA_FULL["y_ev"].numpy()
for tag, res in [("BASE (sin regularización)", res_base), ("MEJOR (Optuna)", res_best)]:
    y_hat = predict(res["model"], DATA_FULL["X_ev"])
    print("=" * 70)
    print(tag)
    print(classification_report(y_test, y_hat, target_names=class_names, digits=3))

# Para que summarize/plot_group reconozcan estas corridas, "val" = test en este grupo
plot_group(["final_baseline", "final_best"], "Final: base vs. mejor configuración (las curvas 'val' son sobre TEST)")
plot_weight_hist(["final_baseline", "final_best"], "Distribución de |w| - modelo final")
summarize(["final_baseline", "final_best"])

**Referencia de trabajos anteriores:** la regresión softmax lineal había dado `0.6581` en test (con `0.9723` en train).

**Observaciones finales** *(completar)*:
- Accuracy en test del modelo base vs. el regularizado vs. softmax lineal (0.6581): `<completar>`
- ¿Se redujo la brecha train-test? `<completar>`
- Categorías que más mejoraron o empeoraron: `<completar>`

## 12. Discusión: justificación de los fenómenos observados

Esta sección vincula lo medido con el marco teórico visto en clase. Para cada fenómeno hay un **fundamento** ya escrito y un espacio para marcar si **se observó o no** en nuestros resultados (si algo no se cumplió, explicar por qué).

| Fenómeno | Fundamento teórico | ¿Observado? |
|---|---|---|
| L2 achica los pesos pero no los anula | El gradiente `2λw` es proporcional al peso: el decaimiento se frena a medida que `w → 0`. Prior gaussiana. | `<completar>` |
| L1 genera pesos casi nulos (esparsidad) | El subgradiente `λ·sign(w)` es de magnitud constante y empuja a cero aunque el peso ya sea chico. Prior de Laplace. | `<completar>` |
| L1 suma ruido a la curva | El peso oscila alrededor de cero porque `sign(w)` cambia de signo. | `<completar>` |
| λ demasiado grande produce underfitting | La penalidad domina a la loss de datos: el modelo no ajusta ni el train. | `<completar>` |
| Dropout aumenta la loss de train (modo train) y la ruidea | Cada paso optimiza una subred distinta. | `<completar>` |
| Dropout converge más lento y reduce la brecha | Evita la co-adaptación; el modelo completo en `eval()` promedia un ensemble de subredes. | `<completar>` |
| Dropout alto con una capa de 32 neuronas subajusta | Pérdida de capacidad en una capa angosta. | `<completar>` |
| BN tolera `lr` más altos | Normaliza la escala de las activaciones y suaviza la superficie de la loss. | `<completar>` |
| BN converge en menos épocas pero cada época es más lenta | Mejor condicionamiento vs. operaciones y parámetros extra por capa. | `<completar>` |
| BN agrega ruido a las curvas | Las estadísticas de un batch de 32 muestras son una estimación ruidosa. Diferencia `train()` vs. `eval()`. | `<completar>` |
| Con BN, `mean|w|` deja de ser una medida de capacidad | Invarianza de escala de los pesos previos a la normalización. | `<completar>` |
| Combinar Dropout con BN puede no sumar | El Dropout cambia la varianza de las activaciones entre train y eval, que BN estimó con la máscara activa. | `<completar>` |

### Recomendaciones de clase aplicadas
1. **Valores absolutos de los pesos:** `mean|w|`, `max|w|`, % de pesos casi nulos e histograma de `|w|` en cada experimento.
2. **Tiempos de entrenamiento:** segundos por época y épocas a una accuracy objetivo.
3. **Learning rate:** barrido con y sin BN, y `lr` dentro del espacio de Optuna (escala log).
4. **Ruido en las curvas:** métrica explícita de ruido, y diferenciación entre loss de train en modo train vs. eval.

### Limitaciones
- Cada configuración se corrió con **una sola semilla**. Diferencias de uno o dos puntos de accuracy pueden caer dentro de la variabilidad entre semillas: conviene repetir con varias semillas antes de afirmar que una técnica es mejor.
- La arquitectura y el preprocesamiento quedaron fijos. Los resultados podrían cambiar con otra arquitectura.

## Visualización en TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/reg